# 03b: Joren et al. Labels (Colab CPU only)

Judge 1,500 questions with Joren et al. (ICLR 2025, App. C.1) sufficient-context prompt using Groq API.

**Test mode:** N_QUESTIONS = 20 (change to None for full 1,500).

In [ ]:
# TEST RUN: set to 20. For full run, set to None.
N_QUESTIONS = 20

In [ ]:
import os
import sys
import json
import time
from datetime import datetime, timezone
from pathlib import Path
from collections import defaultdict
import re

import pandas as pd
import numpy as np
from scipy.stats import contingency

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f"Running on Colab: {IN_COLAB}")

In [ ]:
# ── Setup: works on a laptop (VS Code) and on Colab ───────────────────────
import os, sys, json, subprocess
ON_COLAB = "google.colab" in sys.modules
if ON_COLAB:
    # Colab: get the code from GitHub (push your latest commit first) and mount Drive
    REPO, BRANCH = "/content/HALT-RAG", "lift_signal"
    if not os.path.exists(REPO):
        subprocess.run(["git", "clone", "-q", "-b", BRANCH,
                        "https://github.com/Shaithra2107/HALT-RAG", REPO], check=True)
    else:
        subprocess.run(["git", "-C", REPO, "pull", "-q"], check=True)
    from google.colab import drive
    drive.mount("/content/drive")
else:
    # laptop: this notebook sits in <repo>/notebooks, so the repo is one level up
    REPO = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, os.path.join(REPO, "src"))

from halt_rag.config import load_config, data_path, data_dir
cfg = load_config(os.path.join(REPO, "config.yaml"))
print("running on:", "Colab" if ON_COLAB else "laptop", "| data folder:", data_dir(cfg))

In [ ]:
# Groq config and paths from config.yaml
print(f"Groq limits: RPM={cfg['judge']['groq_rpm_limit']}, TPM={cfg['judge']['groq_tpm_limit']}, TPD={cfg['judge']['groq_tpd_limit']}")

# Reference config model and limits so we don't repeat the dict access everywhere
GROQ_MODEL = cfg['judge']['groq_model']
GROQ_TEMP = cfg['judge']['groq_temperature']
GROQ_MAX_TOKENS = cfg['judge']['groq_max_tokens']
GROQ_RPM = cfg['judge']['groq_rpm_limit']
GROQ_TPM = cfg['judge']['groq_tpm_limit']
GROQ_TPD = cfg['judge']['groq_tpd_limit']
GROQ_RETRY_WAIT = cfg['judge']['groq_retry_wait_secs']
GROQ_MAX_RETRIES = cfg['judge']['groq_max_retries']

In [ ]:
# Read the Joren et al. prompt template
prompt_path = os.path.join(REPO, 'prompts', 'joren_sufficient_context.txt')
with open(prompt_path) as f:
    prompt_template = f.read()

print(f"Prompt loaded ({len(prompt_template)} chars)")
print(f"Template contains: <question>={('<question>' in prompt_template)}, <context>={('<context>' in prompt_template)}, <TIMESTAMP>={('<TIMESTAMP>' in prompt_template)}, TIMESTAMP={('TIMESTAMP' in prompt_template and 'TIMESTAMP' in prompt_template)}")

# Data paths from config.yaml
signals_path = data_path(cfg, "signals_1500")
checkpoint_path = data_path(cfg, "joren_1500_checkpoint")
output_path = data_path(cfg, "joren_1500")

print(f"Signals CSV: {signals_path}")
print(f"Checkpoint: {checkpoint_path}")
print(f"Output: {output_path}")

In [ ]:
# Load halt_rag_1500_signals.csv
df = pd.read_csv(signals_path)

print(f"Loaded {len(df)} rows from {os.path.basename(signals_path)}")
print(f"Columns: {list(df.columns)}")

# Verify required columns
required_cols = ['id', 'question', 'band'] + [f'ret_passage_{i}' for i in range(1, 6)] + [f'ret_title_{i}' for i in range(1, 6)]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    print(f"WARNING: Missing columns: {missing}")
    print(f"Available: {list(df.columns)}")
else:
    print(f"All required columns present")

# Check for y_cov (for agreement calculation)
has_y_cov = 'y_cov' in df.columns
print(f"Has y_cov: {has_y_cov}")

# Limit to N_QUESTIONS if specified
if N_QUESTIONS is not None:
    df = df.iloc[:N_QUESTIONS].copy()
    print(f"Limited to {len(df)} questions for test run")
else:
    print(f"Running on all {len(df)} questions")

In [ ]:
# Fixed column order for all results DataFrames
RESULT_COLUMNS = ['id', 'question', 'band', 'y_joren', 'raw_output', 'parse_ok', 'model', 'timestamp']
checkpoint_every = 25

results = []
parse_failures = []
sample_outputs = []
api_error_ids = []

print(f"Checkpoint path: {checkpoint_path}")
print(f"Output path: {output_path}")

# Resume from checkpoint if it exists and is non-empty
done_ids = set()
if os.path.exists(checkpoint_path):
    try:
        checkpoint_df = pd.read_csv(checkpoint_path)
        if len(checkpoint_df) > 0 and 'id' in checkpoint_df.columns:
            done_ids = set(checkpoint_df['id'].values)
            results = checkpoint_df.to_dict('records')
            parsed = checkpoint_df[checkpoint_df['parse_ok'] == True]
            sample_outputs = parsed.head(3)['raw_output'].tolist() if len(parsed) > 0 else []
            print(f"Resuming: {len(done_ids)} questions already done")
        else:
            print("Checkpoint file is empty or missing columns, starting fresh")
    except Exception as e:
        print(f"Could not read checkpoint ({e}), starting fresh")
else:
    print("Starting fresh (no checkpoint found)")

# Filter to questions not yet done
df_todo = df[~df['id'].isin(done_ids)].copy()
print(f"Questions to process: {len(df_todo)}")

In [ ]:
# Initialize Groq client
try:
    from groq import Groq
except ImportError:
    print("Installing groq...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "groq", "-q"])
    from groq import Groq

# Get API key from Colab Secrets (never print)
if ON_COLAB:
    from google.colab import userdata
    groq_api_key = userdata.get('GROQ_API_KEY')
else:
    groq_api_key = os.environ.get('GROQ_API_KEY')
    if not groq_api_key:
        raise ValueError("GROQ_API_KEY not found. Set it in environment or Colab Secrets.")

client = Groq(api_key=groq_api_key)
print(f"Groq client initialized")

In [ ]:
# Validate that the configured model is available on Groq
try:
    available = [m.id for m in client.models.list().data]
    if GROQ_MODEL not in available:
        raise SystemExit(
            f"Model '{GROQ_MODEL}' not available on Groq.\n"
            f"Available: {', '.join(sorted(available))}\n"
            f"Update judge.groq_model in config.yaml."
        )
    print(f"Model '{GROQ_MODEL}' confirmed available")
except SystemExit:
    raise
except Exception as e:
    print(f"WARNING: could not verify model ({e}), proceeding anyway")

In [ ]:
class RateLimiter:
    """Tracks requests and tokens in a rolling 60-second window."""
    def __init__(self, rpm, tpm, tpd):
        self.rpm = rpm
        self.tpm = tpm
        self.tpd = tpd
        self._req_times = []
        self._tok_log = []
        self.tokens_today = 0

    def wait_if_needed(self, est_tokens):
        """Block until the next request fits within RPM / TPM. Return False if TPD exceeded."""
        if self.tokens_today + est_tokens > self.tpd:
            return False
        while True:
            now = time.time()
            cutoff = now - 60
            self._req_times = [t for t in self._req_times if t > cutoff]
            self._tok_log = [(t, n) for t, n in self._tok_log if t > cutoff]
            waits = []
            if len(self._req_times) >= self.rpm:
                waits.append(self._req_times[0] - cutoff)
            window_tokens = sum(n for _, n in self._tok_log)
            if window_tokens + est_tokens > self.tpm:
                needed = window_tokens + est_tokens - self.tpm
                acc = 0
                for t, n in sorted(self._tok_log):
                    acc += n
                    if acc >= needed:
                        waits.append(t - cutoff)
                        break
            if not waits:
                break
            wait = max(0.1, max(waits))
            print(f"  Rate limit: waiting {wait:.1f}s")
            time.sleep(wait)
        return True

    def record(self, total_tokens):
        now = time.time()
        self._req_times.append(now)
        self._tok_log.append((now, total_tokens))
        self.tokens_today += total_tokens


def estimate_tokens(text):
    return len(text) // 4 + 1


def format_prompt(template, question, passages, titles, timestamp_str):
    context_parts = []
    for title, passage in zip(titles, passages):
        context_parts.append(f"Title: {title} / Text: {passage}")
    context_str = "\n".join(context_parts)
    prompt = template.replace('<question>', question)
    prompt = prompt.replace('<context>', context_str)
    prompt = prompt.replace('<TIMESTAMP>', timestamp_str)
    prompt = prompt.replace('TIMESTAMP', timestamp_str)
    if '<question>' in prompt or '<context>' in prompt or '<TIMESTAMP>' in prompt:
        return None, "Unfilled placeholder remains"
    return prompt, None


def parse_sufficient_context(raw_output):
    match = re.search(r'"Sufficient Context"\s*:\s*([01])', raw_output)
    if match:
        return int(match.group(1)), True
    try:
        obj = json.loads(raw_output)
        if 'Sufficient Context' in obj:
            val = obj['Sufficient Context']
            if val in [0, 1]:
                return val, True
    except Exception:
        pass
    return None, False


def _is_daily_limit(error_str):
    low = error_str.lower()
    return any(kw in low for kw in ['per day', 'tpd', 'rpd', 'daily', 'quota'])


def call_groq(client, prompt, model, temperature, max_tokens, limiter, retry_wait_secs, max_retries):
    est = estimate_tokens(prompt) + max_tokens
    if not limiter.wait_if_needed(est):
        return None, False, "daily token limit reached (estimated)", None
    retry = 0
    while retry < max_retries:
        try:
            resp = client.chat.completions.create(
                model=model,
                messages=[{'role': 'user', 'content': prompt}],
                temperature=temperature,
                max_tokens=max_tokens,
            )
            actual = resp.usage.total_tokens if resp.usage else est
            limiter.record(actual)
            return resp.choices[0].message.content, True, None, resp.usage
        except Exception as e:
            err = str(e)
            if '429' in err:
                if _is_daily_limit(err):
                    return None, False, "daily limit reached", None
                retry_after = None
                ra_match = re.search(r'retry.after[^\d]*(\d+\.?\d*)', err, re.IGNORECASE)
                if ra_match:
                    retry_after = float(ra_match.group(1))
                wait = retry_after if retry_after else retry_wait_secs * (2 ** retry)
                print(f"  429: waiting {wait:.1f}s (retry {retry+1}/{max_retries})")
                time.sleep(wait)
                retry += 1
            else:
                return None, False, f"API error: {err[:200]}", None
    return None, False, f"max retries ({max_retries}) exceeded", None


print("Defined: RateLimiter, estimate_tokens, format_prompt, parse_sufficient_context, call_groq")

In [ ]:
# Main loop
limiter = RateLimiter(GROQ_RPM, GROQ_TPM, GROQ_TPD)
print(f"\nProcessing {len(df_todo)} questions...\n")

daily_limit_hit = False
timestamp_str = '2023-11-01'

for idx, (_, row) in enumerate(df_todo.iterrows(), 1):
    q_id = row['id']
    question = row['question']
    band = row['band']

    passages = [row.get(f'ret_passage_{i}', '') for i in range(1, 6)]
    titles = [row.get(f'ret_title_{i}', '') for i in range(1, 6)]

    prompt, format_err = format_prompt(prompt_template, question, passages, titles, timestamp_str)
    if prompt is None:
        print(f"[{idx}/{len(df_todo)}] ID {q_id}: format error: {format_err}")
        results.append({
            'id': q_id, 'question': question, 'band': band,
            'y_joren': None, 'raw_output': '', 'parse_ok': False,
            'model': GROQ_MODEL, 'timestamp': datetime.now(timezone.utc).isoformat(),
        })
        continue

    response, success, error, usage = call_groq(
        client, prompt, model=GROQ_MODEL, temperature=GROQ_TEMP,
        max_tokens=GROQ_MAX_TOKENS, limiter=limiter,
        retry_wait_secs=GROQ_RETRY_WAIT, max_retries=GROQ_MAX_RETRIES,
    )

    if not success:
        if 'daily' in error:
            print(f"\n*** DAILY LIMIT REACHED ***")
            print(f"Tokens used today (estimated): {limiter.tokens_today:,}")
            daily_limit_hit = True
            break
        else:
            print(f"[{idx}/{len(df_todo)}] ID {q_id}: {error}")
            api_error_ids.append((q_id, question, band))
            continue

    label, parse_ok = parse_sufficient_context(response)
    if not parse_ok:
        print(f"[{idx}/{len(df_todo)}] ID {q_id}: unparseable (will retry)")
        parse_failures.append((q_id, question, band, response))

    results.append({
        'id': q_id, 'question': question, 'band': band,
        'y_joren': label if parse_ok else None, 'raw_output': response,
        'parse_ok': parse_ok, 'model': GROQ_MODEL,
        'timestamp': datetime.now(timezone.utc).isoformat(),
    })

    if parse_ok and len(sample_outputs) < 3:
        sample_outputs.append(response)

    if idx % 10 == 0:
        parsed_so_far = sum(1 for r in results if r['parse_ok'])
        print(f"[{idx}/{len(df_todo)}] {parsed_so_far} parsed | tokens today ~{limiter.tokens_today:,}")

    if idx % checkpoint_every == 0 and len(results) > 0:
        pd.DataFrame(results, columns=RESULT_COLUMNS).to_csv(checkpoint_path, index=False)
        print(f"  Checkpoint saved: {len(results)} rows")

# Save checkpoint at end of loop
if len(results) > 0:
    pd.DataFrame(results, columns=RESULT_COLUMNS).to_csv(checkpoint_path, index=False)
    print(f"\nCheckpoint saved: {len(results)} rows")

print(f"\nLoop finished. Results: {len(results)}, API errors: {len(api_error_ids)}, Parse failures: {len(parse_failures)}")

In [ ]:
# Retry API errors and parse failures once each
retried = 0

if api_error_ids and not daily_limit_hit:
    print(f"\nRetrying {len(api_error_ids)} API-error questions...")
    still_failed = []
    for q_id, question, band in api_error_ids:
        matches = df[df['id'] == q_id]
        if len(matches) == 0:
            still_failed.append((q_id, question, band))
            continue
        row = matches.iloc[0]
        passages = [row.get(f'ret_passage_{i}', '') for i in range(1, 6)]
        titles = [row.get(f'ret_title_{i}', '') for i in range(1, 6)]
        prompt, fmt_err = format_prompt(prompt_template, question, passages, titles, timestamp_str)
        if prompt is None:
            still_failed.append((q_id, question, band))
            continue
        response, success, error, usage = call_groq(
            client, prompt, model=GROQ_MODEL, temperature=GROQ_TEMP,
            max_tokens=GROQ_MAX_TOKENS, limiter=limiter,
            retry_wait_secs=GROQ_RETRY_WAIT, max_retries=GROQ_MAX_RETRIES,
        )
        if not success:
            if 'daily' in error:
                daily_limit_hit = True
                print(f"  Daily limit hit during retry")
                break
            still_failed.append((q_id, question, band))
            continue
        label, parse_ok = parse_sufficient_context(response)
        results.append({
            'id': q_id, 'question': question, 'band': band,
            'y_joren': label if parse_ok else None, 'raw_output': response,
            'parse_ok': parse_ok, 'model': GROQ_MODEL,
            'timestamp': datetime.now(timezone.utc).isoformat(),
        })
        retried += 1
    api_error_ids = still_failed
    print(f"  Recovered: {retried}, still failed: {len(still_failed)}")

# Retry parse failures: re-call the API (the first output was unparseable)
parse_retried = 0
if parse_failures and not daily_limit_hit:
    print(f"\nRetrying {len(parse_failures)} unparseable questions...")
    still_unparsed = []
    for q_id, question, band, old_output in parse_failures:
        matches = df[df['id'] == q_id]
        if len(matches) == 0:
            continue
        row = matches.iloc[0]
        passages = [row.get(f'ret_passage_{i}', '') for i in range(1, 6)]
        titles = [row.get(f'ret_title_{i}', '') for i in range(1, 6)]
        prompt, fmt_err = format_prompt(prompt_template, question, passages, titles, timestamp_str)
        if prompt is None:
            continue
        response, success, error, usage = call_groq(
            client, prompt, model=GROQ_MODEL, temperature=GROQ_TEMP,
            max_tokens=GROQ_MAX_TOKENS, limiter=limiter,
            retry_wait_secs=GROQ_RETRY_WAIT, max_retries=GROQ_MAX_RETRIES,
        )
        if not success:
            if 'daily' in error:
                daily_limit_hit = True
                print(f"  Daily limit hit during retry")
                break
            continue
        label, parse_ok = parse_sufficient_context(response)
        if parse_ok:
            for r in results:
                if r['id'] == q_id and not r['parse_ok']:
                    r['y_joren'] = label
                    r['raw_output'] = response
                    r['parse_ok'] = True
                    r['timestamp'] = datetime.now(timezone.utc).isoformat()
                    parse_retried += 1
                    break
        else:
            still_unparsed.append((q_id, question, band, response))
    parse_failures = still_unparsed
    print(f"  Fixed: {parse_retried}, still unparseable: {len(still_unparsed)}")

# Save after retries
if len(results) > 0:
    pd.DataFrame(results, columns=RESULT_COLUMNS).to_csv(checkpoint_path, index=False)
    print(f"\nCheckpoint updated after retries: {len(results)} rows")

print(f"Total retried: {retried + parse_retried}")

In [ ]:
# Save final output (never save an empty file)
if len(results) > 0:
    results_df = pd.DataFrame(results, columns=RESULT_COLUMNS)
    results_df.to_csv(output_path, index=False)
    print(f"Saved {len(results_df)} results to {os.path.basename(output_path)}")
    results_df.to_csv(checkpoint_path, index=False)
    print(f"Checkpoint updated")
else:
    print("No results to save (empty run)")
    results_df = pd.DataFrame(columns=RESULT_COLUMNS)

In [ ]:
# Compute agreement with y_cov (if available)
if not has_y_cov:
    print("y_cov not available, skipping agreement calculation")
elif len(results_df) == 0 or results_df['parse_ok'].sum() == 0:
    print("No parsed rows, skipping agreement calculation")
else:
    merged = results_df.merge(df[['id', 'y_cov']], on='id', how='left')
    valid = merged[(merged['y_joren'].notna()) & (merged['y_cov'].notna()) & (merged['parse_ok'] == True)].copy()

    if len(valid) == 0:
        print("No valid rows for agreement (need both y_cov and y_joren parsed)")
    else:
        from sklearn.metrics import confusion_matrix, cohen_kappa_score

        cm = confusion_matrix(valid['y_cov'], valid['y_joren'], labels=[0, 1])
        accuracy = (valid['y_cov'] == valid['y_joren']).sum() / len(valid)
        kappa = cohen_kappa_score(valid['y_cov'], valid['y_joren'])

        print("\n" + "="*60)
        print("Agreement: y_joren vs y_cov")
        print("="*60)
        print(f"\n2x2 Contingency Table (y_cov rows x y_joren cols):")
        print(f"              y_joren=0  y_joren=1")
        print(f"y_cov=0       {cm[0, 0]:6d}     {cm[0, 1]:6d}")
        print(f"y_cov=1       {cm[1, 0]:6d}     {cm[1, 1]:6d}")
        print(f"\nOverall:")
        print(f"  Agreement: {accuracy:.1%}")
        print(f"  Cohen's kappa: {kappa:.3f}")
        print(f"  N: {len(valid)}")

        for band_val in ['head', 'tail']:
            band_valid = valid[valid['band'] == band_val]
            if len(band_valid) > 0:
                acc = (band_valid['y_cov'] == band_valid['y_joren']).sum() / len(band_valid)
                kappa_band = cohen_kappa_score(band_valid['y_cov'], band_valid['y_joren'])
                print(f"\n{band_val.upper()}:")
                print(f"  Agreement: {acc:.1%}")
                print(f"  Cohen's kappa: {kappa_band:.3f}")
                print(f"  N: {len(band_valid)}")

In [ ]:
# Print stats
parsed_count = results_df['parse_ok'].sum()
parse_rate = parsed_count / len(results_df) if len(results_df) > 0 else 0

print("\n" + "="*60)
print("Labeling Stats")
print("="*60)
print(f"Total questions: {len(results_df)}")
print(f"Successfully parsed: {parsed_count} ({parse_rate:.1%})")
print(f"Unparseable: {len(results_df) - parsed_count}")

if parsed_count > 0:
    parsed_df = results_df[results_df['parse_ok'] == True]
    pos_count = (parsed_df['y_joren'] == 1).sum()
    neg_count = (parsed_df['y_joren'] == 0).sum()
    print(f"\nLabels (parsed only):")
    print(f"  Sufficient Context = 1: {pos_count} ({pos_count/len(parsed_df):.1%})")
    print(f"  Sufficient Context = 0: {neg_count} ({neg_count/len(parsed_df):.1%})")
    
    # By band
    for band_val in ['head', 'tail']:
        band_df = parsed_df[parsed_df['band'] == band_val]
        if len(band_df) > 0:
            band_pos = (band_df['y_joren'] == 1).sum()
            print(f"\n{band_val.upper():5s}: sufficient={band_pos}/{len(band_df)} ({band_pos/len(band_df):.1%})")

In [ ]:
print("\nDone.")
if daily_limit_hit:
    print("Daily quota hit. Checkpoint saved. Run this notebook again to resume.")
else:
    print(f"All questions processed. Output: {os.path.basename(output_path)}")